# Getting Started

This notebook provides a quick introduction to the [`pynetdicom`](https://pydicom.github.io/pynetdicom/stable/index.html) package. References to documentation sources are provided within their usage context.

## Set up instructions

Some of the following cells show how to communicate with a Service Class Provider (SCP) running on `localhost:11112`. In order to run these cells, execute the following in a separate terminal window:

```sh
storescp --ae-title dicom-hub.server --verbose --log-level debug 11112
```


In [1]:
STORESCP_HOST = 'localhost'
STORESCP_PORT = 11112
STORESCP_AE_TITLE = 'dicom-hub.server'

STORESCU_AE_TITLE = 'dicom-hub.client'

## Testing

[`pydicom`](https://pydicom.github.io/pydicom/stable/index.html) (installed as a dependency of [`pynetdicom`](https://pydicom.github.io/pynetdicom/stable/index.html)) provides several data sets for use in testing.

See also: [`pydicom` User Guide](https://pydicom.github.io/pydicom/stable/guides/user/index.html)


In [2]:
from pydicom.data import get_testdata_file

dataset = get_testdata_file('CT_small.dcm', read=True)
dataset

Dataset.file_meta -------------------------------
(0002, 0000) File Meta Information Group Length  UL: 192
(0002, 0001) File Meta Information Version       OB: b'\x00\x01'
(0002, 0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002, 0003) Media Storage SOP Instance UID      UI: 1.3.6.1.4.1.5962.1.1.1.1.1.20040119072730.12322
(0002, 0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002, 0012) Implementation Class UID            UI: 1.3.6.1.4.1.5962.2
(0002, 0013) Implementation Version Name         SH: 'DCTOOL100'
(0002, 0016) Source Application Entity Title     AE: 'CLUNIE1'
-------------------------------------------------
(0008, 0005) Specific Character Set              CS: 'ISO_IR 100'
(0008, 0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL']
(0008, 0012) Instance Creation Date              DA: '20040119'
(0008, 0013) Instance Creation Time              TM: '072731'
(0008, 0014) Instance Creator UID                U

## `C-ECHO`

The following cell sends a `C-ECHO` verification request to the peer `storescp` Application Entity (AE).

References:

- [Verification Service Examples | Code Examples](https://pydicom.github.io/pynetdicom/stable/examples/verification.html#c-echo)


In [3]:
from pynetdicom import AE, build_context
from pynetdicom.sop_class import Verification

ae = AE(ae_title=STORESCU_AE_TITLE)
ae.requested_contexts = [build_context(Verification)]
association = ae.associate(STORESCP_HOST, STORESCP_PORT, ae_title=STORESCP_AE_TITLE)  # pyright: ignore[reportUnknownMemberType]

if association.is_established:
    status = association.send_c_echo()
    association.release()
    print(f'C-ECHO status: {status}')
else:
    print('Association failed')

C-ECHO status: (0000, 0900) Status                              US: 0
